# DPO Fine-tuning — Ministral-3-14B-Instruct-2512

**Objectif** : apprendre au modèle à vérifier l'effet d'une action *avant* de l'exécuter, via DPO (Direct Preference Optimization).

- **rejected** : réponse originale du modèle qui a produit une action sans effet (`no_effect`)
- **chosen** : réponse synthétique générée avec une instruction de vérification before/after
- **218 paires** issues des runs locaux EWM (ministral-3:14b)

**Avant de lancer** : fermer CS2 pour récupérer ~5 GB VRAM supplémentaires.

## 0. Vérification GPU + dépendances

In [1]:
import subprocess, torch

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.free,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
print(f'torch version : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    free = torch.cuda.mem_get_info()[0] / 1024**3
    total = torch.cuda.mem_get_info()[1] / 1024**3
    print(f'VRAM libre    : {free:.1f} GB / {total:.1f} GB')
    assert free > 14, f'Pas assez de VRAM ({free:.1f} GB < 14 GB requis) — fermer CS2 d\'abord'

NVIDIA GeForce RTX 3090, 23262 MiB, 24576 MiB
torch version : 2.14.0+cu130
CUDA available: True
VRAM libre    : 22.5 GB / 23.6 GB


In [2]:
import trl, peft, transformers, bitsandbytes
print(f'trl          : {trl.__version__}')
print(f'peft         : {peft.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'bitsandbytes : {bitsandbytes.__version__}')

trl          : 1.12.0
peft         : 0.20.0
transformers : 5.15.1
bitsandbytes : 0.50.2


## 1. Configuration

In [3]:
import os
from pathlib import Path

# ── Chemins ──────────────────────────────────────────────────────────────────
MODEL_ID   = 'mistralai/Ministral-3-14B-Instruct-2512'
DATASET    = Path('../results/dpo_train_dataset_2026-09-21/train.jsonl')
OUTPUT_DIR = Path('../checkpoints/dpo_ministral_20260921')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparamètres ──────────────────────────────────────────────────────────
LORA_RANK       = 16          # taille des matrices LoRA
MAX_SEQ_LEN     = 2048        # longueur max d'un exemple tokenisé
BATCH_SIZE      = 1           # batch par GPU
GRAD_ACCUM      = 8           # effective batch = 8
EPOCHS          = 3
LR              = 5e-5
BETA            = 0.1         # poids de la pénalité KL dans DPO

print(f'Dataset     : {DATASET}')
print(f'Output      : {OUTPUT_DIR}')
print(f'Model       : {MODEL_ID}')

Dataset     : ../results/dpo_train_dataset_2026-09-21/train.jsonl
Output      : ../checkpoints/dpo_ministral_20260921
Model       : mistralai/Ministral-3-14B-Instruct-2512


## 2. Chargement du dataset

In [4]:
import json
from datasets import Dataset

records = [json.loads(l) for l in DATASET.read_text().splitlines() if l.strip()]
dataset = Dataset.from_list(records)
print(f'{len(dataset)} paires DPO chargées')
print(f'Colonnes : {dataset.column_names}')

# Aperçu d'une paire
ex = records[10]
print(f'\nExemple (game={ex["game_id"]}, step={ex["analysis_step"]})')
print(f'  prompt msgs  : {len(ex["prompt"])}')
print(f'  chosen (100) : {ex["chosen"][0]["content"][:100]}')
print(f'  rejected(100): {ex["rejected"][0]["content"][:100]}')

218 paires DPO chargées
Colonnes : ['game_id', 'run', 'analysis_step', 'prompt', 'chosen', 'rejected']

Exemple (game=ar25, step=23)
  prompt msgs  : 2
  chosen (100) : World model update after `DOWN` action:
- **Objects:**
  - **Black (B) L-shaped object (`ID: 5`):**

  rejected(100): World model update after `DOWN` action:
- **Objects:**
  - **Black (B) L-shaped object (`ID: 5`):** 


## 3. Chargement du modèle en 4-bit (QLoRA)

In [5]:
import torch
from transformers import AutoTokenizer, AutoConfig, BitsAndBytesConfig
from transformers.models.mistral3.modeling_mistral3 import Mistral3ForConditionalGeneration
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Chargement tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, fix_mistral_regex=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'  vocab size: {tokenizer.vocab_size}')

# Le checkpoint est stocké en FP8 (FineGrainedFP8Config).
# get_hf_quantizer() lit config.quantization_config depuis notre objet,
# pas depuis le disque. On le supprime pour que pre_quantized=False,
# ce qui permet à BnB 4-bit de s'appliquer sans conflit de classes.
# Les clés FP8 (weight_scale_inv, activation_scale) seront ignorées
# comme "unexpected keys" lors du chargement des poids.
print('Chargement config (suppression FP8)...')
config = AutoConfig.from_pretrained(MODEL_ID)
config.quantization_config = None  # retire la FineGrainedFP8Config stockée

print('Chargement modèle 4-bit (BnB nf4)...')
model = Mistral3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    config=config,
    quantization_config=bnb_config,
    device_map='auto',
    dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

total_params = sum(p.numel() for p in model.parameters()) / 1e9
free_gb = torch.cuda.mem_get_info()[0] / 1024**3
print(f'  paramètres: {total_params:.1f}B')
print(f'  VRAM libre après chargement: {free_gb:.1f} GB')

Chargement tokenizer...
  vocab size: 131072
Chargement config (suppression FP8)...
Chargement modèle 4-bit (BnB nf4)...


Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

[transformers] Mistral3ForConditionalGeneration LOAD REPORT from: mistralai/Ministral-3-14B-Instruct-2512
Key                                                                    | Status     |  | 
-----------------------------------------------------------------------+------------+--+-
model.language_model.layers.{0...39}.self_attn.k_proj.activation_scale | UNEXPECTED |  | 
model.language_model.layers.{0...39}.mlp.gate_proj.activation_scale    | UNEXPECTED |  | 
model.language_model.layers.{0...39}.mlp.down_proj.weight_scale_inv    | UNEXPECTED |  | 
model.language_model.layers.{0...39}.self_attn.o_proj.activation_scale | UNEXPECTED |  | 
model.language_model.layers.{0...39}.self_attn.k_proj.weight_scale_inv | UNEXPECTED |  | 
model.language_model.layers.{0...39}.self_attn.v_proj.activation_scale | UNEXPECTED |  | 
model.language_model.layers.{0...39}.self_attn.q_proj.weight_scale_inv | UNEXPECTED |  | 
model.language_model.layers.{0...39}.mlp.gate_proj.weight_scale_inv    | UNEXPECTED 

  paramètres: 7.6B
  VRAM libre après chargement: 8.9 GB


## 4. Configuration LoRA

In [6]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
)
print(f'LoRA rank={LORA_RANK}, alpha={LORA_RANK*2}')
print(f'Modules cibles: {lora_config.target_modules}')

LoRA rank=16, alpha=32
Modules cibles: {'up_proj', 'k_proj', 'gate_proj', 'o_proj', 'down_proj', 'q_proj', 'v_proj'}


## 5. Entraînement DPO

In [7]:
import os, shutil
from trl import DPOConfig, DPOTrainer

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Supprimer le checkpoint précédent (entraîné sur 16 exemples seulement)
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True)
    print(f'Checkpoint précédent supprimé: {OUTPUT_DIR}')

dpo_config = DPOConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=8,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    beta=BETA,
    max_length=2048,                 # 212/218 exemples (vs 16/218 avec 1024)
    precompute_ref_log_probs=True,   # ref log-probs 1x → évite double forward OOM
    dataset_num_proc=4,
    report_to='none',
)

trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Paramètres entraînables: {trainable/1e6:.1f}M / {total/1e9:.1f}B ({100*trainable/total:.2f}%)')

Checkpoint précédent supprimé: ../checkpoints/dpo_ministral_20260921


Tokenizing train dataset (num_proc=4):   0%|          | 0/218 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset (num_proc=4):   0%|          | 0/218 [00:00<?, ? examples…

Computing reference log probs for train dataset:   0%|          | 0/212 [00:00<?, ?it/s]

Caching reference log probs for train dataset:   0%|          | 0/212 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/212 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/212 [00:00<?, ? examples/s]

Paramètres entraînables: 70.0M / 7.7B (0.91%)


In [ ]:
import time
t0 = time.time()
print('Début de l\'entraînement DPO...')
trainer.train()
elapsed = time.time() - t0
print(f'\nTerminé en {elapsed/3600:.1f}h ({elapsed/60:.0f} min)')

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 11}.


Début de l'entraînement DPO...


[W921 23:21:06.639531108 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1073741824 bytes (free: 193789952, total: 25288769536).
[W921 23:21:06.661309722 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1073741824 bytes (free: 462225408, total: 25288769536).


OutOfMemoryError: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 23.55 GiB of which 440.88 MiB is free. Process 3037 has 262.00 MiB memory in use. Including non-PyTorch memory, this process has 22.31 GiB memory in use. Of the allocated memory 20.66 GiB is allocated by PyTorch, and 1.34 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

: 

## 6. Sauvegarde de l'adaptateur LoRA

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

adapter_files = list(OUTPUT_DIR.glob('adapter*'))
print(f'Adaptateur sauvegardé dans: {OUTPUT_DIR}')
for f in adapter_files:
    print(f'  {f.name}: {f.stat().st_size/1024**2:.1f} MB')

Adaptateur sauvegardé dans: ../checkpoints/dpo_ministral_20260921
  adapter_model.safetensors: 133.6 MB
  adapter_config.json: 0.0 MB


## 7. Vérification rapide — génération avant/après

Test sur un exemple du dataset : est-ce que le modèle fine-tuné génère maintenant une vérification avant d'appeler `action()` ?

In [ ]:
import torch

# trainer.model EST déjà le PeftModel fine-tuné — ne pas recharger via
# PeftModel.from_pretrained(model, ...) qui ajouterait un 2e adapter
# et provoquerait des NaN dans les probabilités de génération.
ft_model = trainer.model
ft_model.eval()

ex = records[5]
messages = ex['prompt']   # list of {role, content}

input_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
# Tronquer le prompt si trop long (sinon OOM sur la génération)
inputs = tokenizer(input_text, return_tensors='pt',
                   truncation=True, max_length=900).to('cuda')

print(f'Prompt tokens: {inputs["input_ids"].shape[1]}')
with torch.no_grad():
    out = ft_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,        # greedy — évite NaN softmax sur modèle 4-bit
    )

response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== Réponse fine-tunée (greedy, 600 chars) ===')
print(response[:600])
print()
print('=== Original rejeté (300 chars) ===')
print(ex['rejected'][0]['content'][:300])

[transformers] Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt tokens: 900
=== Réponse fine-tunée (greedy, 600 chars) ===


=== Original rejeté (300 chars) ===
**World Model Update:**
- The light green object (N) is **not present** in the current segmentation, suggesting it may have been misidentified or removed from the board. The current segmentation shows the following:
  - **Black (B) and blue (b) objects** are present, including one black object align


: 

## 8. Test live — adapter vs baseline dans le jeu

Lance le `ReplToolsAgent` sur quelques parties avec et sans l'adapter DPO,
et mesure le taux d'actions sans effet (`no_effect_rate`).

**Runtime** : ~10 min par game × 2 conditions × 2 games ≈ 40 min.  
Peut être réduit en baissant `MAX_ACTIONS` ou en n'utilisant qu'un seul game.

In [1]:
# ── 8a. Chargement du modèle fine-tuné (autonome — pas besoin d'avoir lancé
#        les cellules précédentes dans cette session de kernel) ─────────────
import sys, os, torch
from pathlib import Path

REPO = Path('..').resolve()
for _p in [str(REPO / 'src'), str(REPO / 'data/ARC-AGI-3-Agents')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)
os.environ.setdefault('OPERATION_MODE',  'offline')
os.environ.setdefault('ENVIRONMENTS_DIR', str(REPO / 'data/environment_files'))
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from transformers import AutoTokenizer, BitsAndBytesConfig
from transformers.models.mistral3.modeling_mistral3 import Mistral3ForConditionalGeneration
from peft import PeftModel

BF16_EXPORT = REPO / 'checkpoints/ministral_bf16_export'
ADAPTER_DIR = REPO / 'checkpoints/dpo_ministral_20260921'

assert BF16_EXPORT.exists(), 'Run notebooks/export_bf16.py first'
assert (ADAPTER_DIR / 'adapter_model.safetensors').exists(), 'DPO adapter not found'

print('Loading fine-tuned model (BF16 export + NF4 + DPO adapter)...')
_bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
_base = Mistral3ForConditionalGeneration.from_pretrained(
    str(BF16_EXPORT), quantization_config=_bnb, device_map='auto',
    torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
)
_ft_model = PeftModel.from_pretrained(_base, str(ADAPTER_DIR))
_ft_model.eval()
_tokenizer = AutoTokenizer.from_pretrained(str(BF16_EXPORT), fix_mistral_regex=True)
if _tokenizer.pad_token is None:
    _tokenizer.pad_token = _tokenizer.eos_token

free_gb = torch.cuda.mem_get_info()[0] / 1024**3
print(f'Ready. VRAM free: {free_gb:.1f} GB  '
      f'(adapter_size: {(ADAPTER_DIR / "adapter_model.safetensors").stat().st_size / 1024**2:.0f} MB)')

Loading fine-tuned model (BF16 export + NF4 + DPO adapter)...


Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

Ready. VRAM free: 13.3 GB  (adapter_size: 116 MB)


In [2]:
# ── 8b. Monkey-patch _query_llm pour utiliser le modèle local ───────────────
import llm_repl_agent

def _make_local_query(use_adapter: bool):
    """Retourne une fonction _query_llm qui utilise le modèle chargé localement."""
    def _fn(prompt: str, model_name=None) -> str:
        if use_adapter:
            _ft_model.enable_adapter_layers()
        else:
            _ft_model.disable_adapter_layers()
        messages = [
            {'role': 'system', 'content': llm_repl_agent.SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ]
        text = _tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = _tokenizer(
            text, return_tensors='pt', truncation=True, max_length=3072
        ).to(_ft_model.device)
        with torch.no_grad():
            out = _ft_model.generate(**inputs, max_new_tokens=400, do_sample=False)
        return _tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return _fn

# Patch initial (sera modifié dans run_game selon la condition)
llm_repl_agent._query_llm = _make_local_query(use_adapter=True)
print('_query_llm patched → local model (adapter toggle via _make_local_query)')

_query_llm patched → local model (adapter toggle via _make_local_query)


In [4]:
# ── 8c. Run live games — fine-tuned vs baseline ─────────────────────────────
import time, numpy as np
import arc_agi
from llm_repl_agent import ReplToolsAgent

GAMES       = ['ft09', 'ar25']  # games avec le plus de no_effect dans le dataset DPO
MAX_ACTIONS = 30                 # ~10 min/game — baisser à 15 pour un test rapide

def _no_effect_count(frames) -> int:
    """Frames consécutives identiques = action sans effet.
    Ignore les FrameData vides (frame=[] - dummy init frame)."""
    valid = [f for f in frames if f.frame]
    n = 0
    for i in range(1, len(valid)):
        prev = np.array(valid[i-1].frame[0])
        curr = np.array(valid[i].frame[0])
        if np.array_equal(prev, curr):
            n += 1
    return n

def run_game(game_id: str, use_adapter: bool) -> dict:
    cond = 'fine-tuned' if use_adapter else 'baseline'
    print(f'\n>>> {game_id} [{cond}]', flush=True)
    llm_repl_agent._query_llm = _make_local_query(use_adapter=use_adapter)

    arcade = arc_agi.Arcade()
    env    = arcade.make(game_id)
    env.reset()
    agent  = ReplToolsAgent(
        card_id='live-eval', game_id=game_id, agent_name='live-eval',
        ROOT_URL='http://offline', record=False, arc_env=env, tags=[],
    )
    agent.MAX_ACTIONS = MAX_ACTIONS

    t0 = time.time()
    try:
        agent.main()
    except Exception as e:
        print(f'  agent error: {e}')

    elapsed  = time.time() - t0
    latest   = agent.frames[-1] if agent.frames else None
    no_eff   = _no_effect_count(agent.frames)
    n_act    = agent.action_counter

    row = dict(
        game        = game_id,
        condition   = cond,
        actions     = n_act,
        levels      = int(latest.levels_completed) if latest else 0,
        no_effect   = no_eff,
        no_eff_pct  = f'{100 * no_eff / max(1, n_act):.0f}%',
        brain_calls = getattr(agent, '_brain_calls', '?'),
        elapsed_min = f'{elapsed/60:.1f}m',
    )
    print(f'  actions={row["actions"]}  levels={row["levels"]}  '
          f'no_effect={row["no_effect"]} ({row["no_eff_pct"]})  '
          f'brain_calls={row["brain_calls"]}  time={row["elapsed_min"]}',
          flush=True)
    return row

live_results = []
for _game in GAMES:
    live_results.append(run_game(_game, use_adapter=True))   # fine-tuned
    live_results.append(run_game(_game, use_adapter=False))  # baseline


>>> ft09 [fine-tuned]
INFO:arc_agi.scorecard:Initialized ScorecardManager with idle_for=0:15:00 and max_open_for=3 days, 0:00:00
2026-09-22 15:16:20 | INFO | Created new scorecard: a0f1a24d-6f00-48f2-8767-efb2873d1557
2026-09-22 15:16:20 | INFO | Found latest version of ft09: ft09-0d8bbf25 (downloaded: 2026-04-08 20:07:48.508616+00:00)
[ReplToolsAgent] code exec failed: NameError: name 'background_colors' is not defined
  actions=31  levels=0  no_effect=30 (97%)  brain_calls=49  time=35.0m

>>> ft09 [baseline]
INFO:arc_agi.scorecard:Initialized ScorecardManager with idle_for=0:15:00 and max_open_for=3 days, 0:00:00
2026-09-22 15:51:23 | INFO | Created new scorecard: 7e3d7c6b-eefc-4b62-82f2-dc64f00aa4b0
2026-09-22 15:51:23 | INFO | Found latest version of ft09: ft09-0d8bbf25 (downloaded: 2026-04-08 20:07:48.508616+00:00)
  actions=31  levels=0  no_effect=30 (97%)  brain_calls=40  time=15.3m

>>> ar25 [fine-tuned]
INFO:arc_agi.scorecard:Initialized ScorecardManager with idle_for=0:15:00

In [ ]:
# ── 8d. Tableau comparatif ───────────────────────────────────────────────────
W = 10
print(f'\n{"game":<8} {"condition":<12} {"actions":>{W}} {"levels":>{W}} '
      f'{"no_effect":>{W}} {"no_eff%":>{W}} {"time":>{W}}')
print('-' * (8 + 12 + 5 * (W + 1)))
for r in live_results:
    print(f'{r["game"]:<8} {r["condition"]:<12} {r["actions"]:>{W}} '
          f'{r["levels"]:>{W}} {r["no_effect"]:>{W}} '
          f'{r["no_eff_pct"]:>{W}} {r["elapsed_min"]:>{W}}')

# Résumé par condition
ft_rows   = [r for r in live_results if r['condition'] == 'fine-tuned']
base_rows = [r for r in live_results if r['condition'] == 'baseline']
def _mean_eff(rows):
    return sum(r['no_effect'] for r in rows) / max(1, sum(r['actions'] for r in rows))
print(f'\nno_effect global — fine-tuned: {_mean_eff(ft_rows):.1%}   '
      f'baseline: {_mean_eff(base_rows):.1%}')